## Importing packages and the scripts

In [ ]:
import matplotlib as mpl
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import seaborn as sns
from google.colab import files
# uploaded the csv from local computer

In [ ]:
files.upload()
df = pd.read_csv("/content/final_round2.csv")
df.columns

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

### Display the Transaction Start Date Summary



In [ ]:
df["Transaction Start Date"].value_counts()

### Visualization of Transaction Start Date Years

In [ ]:
# Extract year
df['Transaction Year'] = pd.to_datetime(df['Transaction Start Date'], errors='coerce').dt.year

# Count rows per year
year_counts = df['Transaction Year'].value_counts().sort_index()

# Plot
plt.figure(figsize=(9, 5))
ax = sns.barplot(x=year_counts.index, y=year_counts.values, color="#4C72B0")  # Single color

# Add labels to all bars
for container in ax.containers:
    ax.bar_label(container, padding=2)
plt.xlabel("Year")
plt.ylabel("Number of Transactions")
plt.title("Transactions per Year")
plt.tight_layout()
plt.show()

### FULL/LITE Documentation: Number of Transactions

In [ ]:
doc_counts = df['Full/Lite Documentation'].value_counts().reset_index()
plt.figure(figsize=(9, 5))
bars = plt.bar(doc_counts['Full/Lite Documentation'], doc_counts['count'])
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height, f'{int(height)}', ha='center', va='bottom')

plt.xlabel("Documentation Type")
plt.ylabel("Number of Transactions")
plt.title("Transactions by Documentation Type")
plt.tight_layout()
plt.show()


### MSAs

In [ ]:
!pip install plotly

In [ ]:
print(df['MSA'].value_counts())

In [ ]:


msa_counts = df['MSA'].value_counts()

ax = msa_counts.plot(kind='barh', figsize=(10, 8), color='#4C72B0')
plt.xlabel("Number of Transactions")
plt.ylabel("MSA")
plt.title("Number of Transactions for MSAs")

# Add value labels inside bars
for i, val in enumerate(msa_counts.values):
    ax.text(val - max(msa_counts.values)*0.02, i, f'{val}', va='center', ha='right', color='white', fontweight='bold')

plt.tight_layout()
plt.show()


### Number of bedrooms X MSAs

In [ ]:
# Define color mapping
bedroom_colors = {
    '1': '#4C72B0',  # blue
    '2': '#55A868',  # green
    '3': '#FFA500'   # orange
}

# Get unique MSAs
grouped = df.groupby(['MSA', 'Bedrooms']).size().to_frame(name='Transaction Count').reset_index()
grouped['Bedrooms'] = grouped['Bedrooms'].astype(str)

unique_msas = grouped['MSA'].unique()

# Loop and create a pie chart for each MSA
for msa in unique_msas:
    subset = grouped[grouped['MSA'] == msa].copy()

    # # Sort for consistent order
    # subset = subset.sort_values('Bedrooms')

    # Convert Bedrooms to string if not already
    subset['Bedrooms'] = subset['Bedrooms'].astype(str)

    # Assign colors based on Bedrooms
    colors = [bedroom_colors.get(b, '#CCCCCC') for b in subset['Bedrooms']]  # default to gray if missing

    # Plot pie chart
    plt.figure(figsize=(5, 5))
    plt.pie(
        subset['Transaction Count'],
        labels=subset['Bedrooms'],
        colors=colors,
        autopct='%1.1f%%',
        startangle=90
    )
    plt.title(f'Bedroom Distribution in {msa}')
    plt.tight_layout()
    # plt.savefig(f"test_bedrooms.png", dpi = 250)
    plt.show()



### Age Req X MSAs

In [ ]:
df['Age Restricted'].value_counts()

In [ ]:
# Define color mapping
restriction_colors = {
    '55+': '#4C72B0',  # blue
    'No': '#55A868',   # green
    '62+': '#FFA500'   # orange
}

# Strip whitespace and standardize casing
df['MSA'] = df['MSA'].str.strip().str.upper()
# Normalize dashes to hyphen-minus
df['MSA'] = df['MSA'].str.replace('–', '-', regex=False)
df['MSA'] = df['MSA'].str.strip().str.upper()

print(df['MSA'].value_counts())


grouped = df.groupby(['MSA', 'Age Restricted']).size().to_frame(name='Transaction Count').reset_index()
grouped['Age Restricted'] = grouped['Age Restricted'].astype(str)

# Get only unique MSAs
unique_msas = grouped['MSA'].unique()
# Loop through each unique MSA
for msa in unique_msas:
    subset = grouped[grouped['MSA'] == msa].copy()

    # Assign colors
    colors = [restriction_colors.get(value, '#CCCCCC') for value in subset['Age Restricted']]

    # Plot
    plt.figure(figsize=(5, 5))
    plt.pie(
        subset['Transaction Count'],
        labels=subset['Age Restricted'],
        colors=colors,
        autopct='%1.1f%%',
        startangle=90
    )
    plt.title(f'Age Restriction Distribution in {msa}')
    plt.tight_layout()
    plt.show()


## Number of Bedrooms

In [ ]:
df["Bedrooms"].value_counts()

# Number of Bedrooms X Age Restriction

In [ ]:
# Clean and prepare the data
df['Bedrooms'] = df['Bedrooms'].astype(str)
df['Age Restricted'] = df['Age Restricted'].astype(str)

# Group and count
bed_age_counts = df.groupby(['Bedrooms', 'Age Restricted']).size().to_frame(name='Transaction Count').reset_index()


# Define consistent colors for age restrictions
restriction_colors = {
    '55+': '#4C72B0',  # blue
    'No': '#55A868',   # green
    '62+': '#FFA500'   # orange
}

# Plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=bed_age_counts,
    x='Bedrooms',
    y='Transaction Count',
    hue='Age Restricted',
    palette=restriction_colors
)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, padding=3)

plt.title("Transactions by Bedrooms and Age Restriction")
plt.xlabel("Number of Bedrooms")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()


# Property MSA frequency X FULL/LITE Documentation

In [ ]:
# Step 1: Group and count
msa_doc_counts = df.groupby(['MSA', 'Full/Lite Documentation']).size().to_frame(name='Transaction Count').reset_index()

# Step 2: Compute total transactions per MSA and use it to sort
msa_totals = msa_doc_counts.groupby('MSA')['Transaction Count'].sum().sort_values(ascending=False)
ordered_msas = msa_totals.index.tolist()

# Step 3: Convert 'MSA' to ordered categorical to control the plot order
msa_doc_counts['MSA'] = pd.Categorical(msa_doc_counts['MSA'], categories=ordered_msas, ordered=True)

# Step 4: Plot horizontal bar chart
plt.figure(figsize=(10, 8))
ax = sns.barplot(
    data=msa_doc_counts,
    y='MSA',
    x='Transaction Count',
    hue='Full/Lite Documentation',
    palette={'FULL': '#4C72B0', 'LITE': '#55A868'}
)

# Add labels
for container in ax.containers:
    ax.bar_label(container, padding=3)

plt.ylabel("MSA")
plt.xlabel("Number of Transactions")
plt.title("Transactions by Documentation Type and MSA (Sorted)")
plt.tight_layout()
plt.show()


# Base Question 05

In [ ]:
df['MSA'].value_counts()

In [ ]:
print("The percentage of properties by MSAs:")
msa_counts = df['MSA'].value_counts(normalize=True).mul(100).round(1)
msa_counts = msa_counts.reset_index()
plt.figure(figsize=(16, 10))
ax = sns.barplot(data=msa_counts, y='MSA', x='proportion', color='#4C72B0')
for i, val in enumerate(msa_counts['proportion']):
    ax.text(val + 0.5, i, f'{val}%', va='center')

plt.xlabel("Percentage of Properties")
plt.ylabel("MSA")
plt.title("Percentage of Properties by MSA")
plt.tight_layout()
plt.savefig(f"msas_properties.png", dpi = 250)
plt.show()
